#### Usosweb documents scraping

In [ ]:
from USOSDataLoader import USOSDataLoader

loader = USOSDataLoader()
documents = loader.get_documents()

#### Knowledge base generation

In [ ]:
from dotenv import find_dotenv, load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv(find_dotenv())

In [ ]:
import os

from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

embeddings = HuggingFaceEmbeddings(model_name="jinaai/jina-embeddings-v3",
                                   model_kwargs={"trust_remote_code": True},
                                   encode_kwargs={"task": "retrieval.query"})

pc = Pinecone(os.environ.get("PINECONE_API_KEY"))

In [ ]:
import time

from pinecone import ServerlessSpec

index_name = "usos-bot"
existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

if index_name not in existing_indexes:
    pc.create_index(
        name=index_name,
        dimension=1024,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )
    while not pc.describe_index(index_name).status["ready"]:
        time.sleep(1)

index = pc.Index(index_name)

vectorstore = PineconeVectorStore(index=index, embedding=embeddings)

In [ ]:
from uuid import uuid4

from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)
uuids = [str(uuid4()) for _ in range(len(splits))]
vectorstore.add_documents(documents=splits, ids=uuids)